# 8주차 — DML — 입력·수정·삭제와 트랜잭션
**의료진을 위한 데이터베이스** · 미니 EMR 실습 노트북

사용법: 위에서부터 순서대로 셀을 실행하세요 (`Shift+Enter`).
`q("SELECT ...")` 안의 SQL만 바꿔가며 실험하면 됩니다. 원본은 언제든 다시 내려받을 수 있으니 마음껏 실험하세요.

In [ ]:
# [준비] 미니 EMR 데이터베이스 내려받기 + 조회 도우미 정의 — 이 셀을 가장 먼저 실행!
import os, urllib.request, sqlite3
import pandas as pd

DB_URL = "https://raw.githubusercontent.com/kevinkwshin/medical-db-course-2026/main/mini_emr/mini_emr.sqlite"   # 강의자 공지 주소
if not os.path.exists("mini_emr.sqlite"):
    try:
        urllib.request.urlretrieve(DB_URL, "mini_emr.sqlite")
        print("mini_emr.sqlite 다운로드 완료")
    except Exception as e:
        print("자동 다운로드 실패 — 왼쪽 파일 탭에서 mini_emr.sqlite를 직접 업로드하세요.", e)

con = sqlite3.connect("mini_emr.sqlite")

def q(sql):
    """SQL을 실행하고 결과를 표로 반환"""
    return pd.read_sql_query(sql, con)

def run(sql):
    """결과를 반환하지 않는 SQL(INSERT/UPDATE/CREATE 등) 실행"""
    con.executescript(sql); con.commit(); print("실행 완료")

q("SELECT name AS 테이블 FROM sqlite_master WHERE type='table' ORDER BY 1")

## 1. INSERT

In [ ]:
# 가상 환자 P051 등록
run("""
INSERT INTO patient (patient_id, sex, birth_date, blood_type)
VALUES ('P051', 'F', '1988-03-15', 'A+');
""")

In [ ]:
# 확인
q("""
SELECT * FROM patient WHERE patient_id = 'P051'
""")

## 2. UPDATE — WHERE 확인 습관

In [ ]:
# ① 먼저 SELECT로 대상 확인
q("""
SELECT lab_id, test_name, result_value FROM lab WHERE lab_id = 'L0044'
""")

In [ ]:
# ② 그 다음 UPDATE
run("""
UPDATE lab SET result_value = 176 WHERE lab_id = 'L0044';
""")

In [ ]:
# ③ 결과 재확인
q("""
SELECT lab_id, test_name, result_value FROM lab WHERE lab_id = 'L0044'
""")

## 3. 트랜잭션 — 없던 일로 만들기

In [ ]:
# BEGIN → UPDATE → ROLLBACK 실습
cur = con.cursor()
cur.execute("BEGIN")
cur.execute("UPDATE lab SET result_value = 0")   # 끔찍한 실수!
print('실수 직후:', cur.execute("SELECT COUNT(*) FROM lab WHERE result_value = 0").fetchone())
con.rollback()                                      # 되돌리기
print('롤백 후  :', cur.execute("SELECT COUNT(*) FROM lab WHERE result_value = 0").fetchone())

## 4. soft delete + 감사로그 패턴

In [ ]:
# 감사로그 테이블과 정정 이력 남기기
run("""
CREATE TABLE IF NOT EXISTS lab_audit (
  audit_id INTEGER PRIMARY KEY,
  lab_id TEXT, old_value REAL, new_value REAL,
  changed_by TEXT, changed_at DATETIME, reason TEXT
);
INSERT INTO lab_audit (lab_id, old_value, new_value, changed_by, changed_at, reason)
VALUES ('L0044', 167, 176, 'S001', '2026-04-02 09:30', '재검 결과 반영');
""")

In [ ]:
# 감사로그 확인
q("""
SELECT * FROM lab_audit
""")

## 연습문제

In [ ]:
# [연습 Q1] P051의 외래 내원 한 건을 visit에 INSERT 하라 (visit_id는 'V0101' 등 새 값)
q("""
-- 여기에 SQL을 작성하세요

""")

In [ ]:
# [연습 Q2] P051의 혈액형을 B+로 정정하라 (SELECT → UPDATE → SELECT 3단계로)
q("""
-- 여기에 SQL을 작성하세요

""")

In [ ]:
# [연습 Q3] (사본이니 안심하고) WHERE 없는 UPDATE를 실행해 피해 규모를 확인한 뒤, 데이터를 다시 내려받아 복구하라
q("""
-- 여기에 SQL을 작성하세요

""")

In [ ]:
# [연습 Q4] lab_audit에 '단위 오입력 정정' 이력을 한 건 더 남겨 보라
q("""
-- 여기에 SQL을 작성하세요

""")

---
## 정답 및 해설
먼저 스스로 푼 다음에 열어 보세요.

**A1. P051 내원**
```sql
INSERT INTO visit (visit_id, patient_id, visit_date, department, visit_type)
VALUES ('V0101', 'P051', '2026-04-01', '가정의학과', '외래');
```

**A2. 혈액형 정정**
```sql
SELECT * FROM patient WHERE patient_id='P051';
UPDATE patient SET blood_type='B+' WHERE patient_id='P051';
SELECT * FROM patient WHERE patient_id='P051';
```